# 📘 Notebook 17: Machine Learning with Spark MLlib

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 5 of 6 in this module · (17 of 18 overall)*

---

> 🧭 **How to use this notebook.** Run the cells in order, from top to bottom: click a cell and press **Shift+Enter**. After most cells there is a short paragraph headed **What you should see**: compare it with your own output before you move on. If your numbers differ, stop and find out why, because that is how data work is learned. The exercises at the end start easy and get harder: first **warm-up** questions, then **think it through** questions (what would you run, and why?), then **technical** tasks. Try each one before you open its solution.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Turn a warehouse into a **training table**: one row per customer, a label, and features that do not leak the answer
- Build an MLlib **pipeline** of transformers and an estimator, and say why pipelines matter on a cluster
- Evaluate a classifier honestly: a **baseline**, a held-out **test set**, the **confusion matrix** and the **AUC**
- Compare two models, and read which features a tree relies on
- Segment customers with **k-means** and choose the number of clusters with the **silhouette**

> **Prerequisites:** Notebook 16 (this data in Spark). The ideas of machine learning are taught in depth in *From Python to Deep Learning and Fine-Tuning*, Notebooks 08 to 11; this notebook explains in plain words what it needs and shows how the ideas run when the data is in Spark.
>
> 🔭 **Why this notebook matters:** a model trained on a laptop with scikit-learn needs all the training data in one machine's memory. When the data lives in a cluster, the training must go to the data. Spark's machine learning library, **MLlib**, provides the familiar algorithms in a form that runs on every machine of the cluster, organised as **pipelines**: the same preparation steps, fitted once on the training data and applied identically to any new data. The modelling questions stay the same; what changes is where the work happens and how carefully the steps are chained.

> 📦 **Libraries used in this notebook.** **PySpark** with its **MLlib** module, in local mode, and **pandas** and **Matplotlib** for small results. The cell below installs PySpark; in Google Colab it is already present.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
%pip install -q pyspark

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

In [ ]:
import sys

import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.functions import vector_to_array

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
spark = (SparkSession.builder.master("local[*]").appName("adventureworks-ml")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

**What you should see:** the ten table names, then `Spark 3.x.x`.

## 1. Machine learning in plain words

A **classifier** learns to answer a yes-or-no question from examples. You give it many rows, each with:

* **features**: what we know about the thing (here: a customer's age, income, family, country ...);
* a **label**: the right answer for that row (here: 1 if the customer bought a bicycle, 0 if not).

During **training** (`fit`), the model adjusts its internal numbers until its answers match the labels as well as it can. Then it can **predict** the label of rows it has never seen (`transform` in Spark). To know whether it really learned, and did not just memorise, we keep some rows aside as a **test set**, never shown during training, and grade the model on them only.

The steps of this notebook follow that plan: build the table of features and labels (Section 2), find out how good pure guessing is (3), set aside a test set (4), prepare the features and train (5), grade (6), compare a second model (7). Section 8 then turns to a question with no labels at all.

## 2. From a warehouse to a training table

### Intuition first
A model learns from **one row per thing we want to predict about**. Our question is the one the company's marketing team would ask: *which customers buy a bicycle?* A customer who is likely to buy one but has not yet is worth a brochure. So the unit is the **customer**, the **label** is 1 if they ever bought a bicycle and 0 otherwise, and the **features** are what the company knows about a person before they buy: age, income, family, education, occupation, home, cars, commute and country.

Notice what is *not* among the features: how much the customer spent, or how many orders they placed. Those are partly caused by buying a bicycle. A model given them would look brilliant in testing and be useless in practice, because for a new prospect they do not exist yet. Using information that would not be available at prediction time is called **leakage**, and it is the most common way that machine learning projects fool themselves (Exercise 8 measures it).

How the code builds the table: find the product keys of all bikes; find the customers who bought one of them (label 1); join every customer to that list with a left join, so that customers who never bought a bike get no match, and `coalesce` turns their missing label into 0.

In [ ]:
def read(name):
    return spark.read.csv(f"{DATA}/{name}.csv", header=True, inferSchema=True)

sales, product = read("FactInternetSales"), read("DimProduct")
subcategory, category = read("DimProductSubcategory"), read("DimProductCategory")
customer, geography = read("DimCustomer"), read("DimGeography")
last_day = sales.agg(F.max(F.to_date(F.substring("OrderDate", 1, 10)))).first()[0]

bike_keys = (product.join(subcategory, "ProductSubcategoryKey").join(category, "ProductCategoryKey")
                    .filter(F.col("EnglishProductCategoryName") == "Bikes").select("ProductKey"))
buyers = (sales.join(bike_keys, "ProductKey").select("CustomerKey").distinct()
               .withColumn("label", F.lit(1.0)))

data = (customer.join(geography.select("GeographyKey", "EnglishCountryRegionName"), "GeographyKey")
        .join(buyers, "CustomerKey", "left")
        .select("CustomerKey",
                F.coalesce("label", F.lit(0.0)).alias("label"),
                F.floor(F.months_between(F.lit(last_day), F.to_date("BirthDate")) / 12).alias("age"),
                F.col("YearlyIncome").cast("double").alias("income"),
                "TotalChildren", "NumberChildrenAtHome", "HouseOwnerFlag", "NumberCarsOwned",
                "Gender", "MaritalStatus", "EnglishEducation", "EnglishOccupation", "CommuteDistance",
                F.col("EnglishCountryRegionName").alias("country")))
data.cache()
data.show(3)

**What you should see:** three customers, one row each, with `label` 1.0 (all three bought a bike), their age (37, 38, 29), income, children, home, cars, and text columns such as `Bachelors`, `Professional`, `5-10 Miles` and `Australia`. This is the whole training table: 18,484 rows, one per customer.

## 3. The baseline: how good is guessing?

Before any model, find out how well a model that ignores every feature would do: always answer the most common label. If a classifier cannot beat that, it has learned nothing.

In [ ]:
balance = data.groupBy("label").count().toPandas().set_index("label")["count"]
print(balance)
print(f"Always guessing the majority class is right {balance.max() / balance.sum():.1%} of the time.")

**What you should see:** 9,132 buyers (label 1.0) and 9,352 non-buyers (label 0.0), and `Always guessing the majority class is right 50.6% of the time.`

The classes are close to balanced, so **accuracy** (the share of right answers) is a fair first measure here, and anything clearly above 50.6% is real learning. (When one class is rare, say 2 percent fraud, a model that always answers *no fraud* is 98 percent accurate and worthless; then precision, recall and the AUC are the measures to watch.)

## 4. A held-out test set

The test set must be put aside **before** anything is fitted, including the scaling of the features. `randomSplit([0.8, 0.2], seed=42)` puts about 80% of the rows in `train` and 20% in `test`; the fixed `seed` makes the split the same every time the notebook runs.

In [ ]:
train, test = data.randomSplit([0.8, 0.2], seed=42)
print("training rows:", train.count(), "  test rows:", test.count())

**What you should see:** about 14,900 training rows and about 3,600 test rows (14,908 and 3,576 on our machine; the exact split can differ slightly between machines).

## 5. An MLlib pipeline

### Intuition first
A model needs numbers, and MLlib wants all the features of a row in **one** column, as a list of numbers called a **vector**. Four preparation steps get us there:

1. `StringIndexer` gives each text value a number (`Bachelors` might become 0, `Graduate Degree` 1, ...);
2. `OneHotEncoder` turns each number into a row of 0s with a single 1, so that the model does not think *Graduate Degree* is "more" than *Bachelors*;
3. `VectorAssembler` gathers all the numeric and encoded columns into one vector column;
4. `StandardScaler` scales each feature so that income (tens of thousands) does not drown out the number of children (0 to 5).

Each of these steps **learns** something from the data (the list of categories, the means and standard deviations), so each must be learned from the **training** data only and then applied unchanged to the test data and to any future customer. A **pipeline** packages the steps with the model, so that this happens automatically and can never be done in the wrong order. On a cluster the pipeline is also what gets shipped to the machines that apply it.

MLlib calls a step that learns from data an **estimator** (it has `fit`), and a step that only transforms a **transformer** (it has `transform`). Fitting a pipeline returns a **pipeline model**, a chain of transformers.

In [ ]:
categorical = ["Gender", "MaritalStatus", "EnglishEducation", "EnglishOccupation", "CommuteDistance", "country"]
numeric = ["age", "income", "TotalChildren", "NumberChildrenAtHome", "HouseOwnerFlag", "NumberCarsOwned"]

indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in categorical]
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in categorical], outputCols=[c + "_vec" for c in categorical])
assembler = VectorAssembler(inputCols=numeric + [c + "_vec" for c in categorical], outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
prepare = indexers + [encoder, assembler, scaler]

logistic = Pipeline(stages=prepare + [LogisticRegression(maxIter=50)]).fit(train)   # learn from train only
predictions = logistic.transform(test)                                            # apply to test
predictions.select("CustomerKey", "label", "probability", "prediction").show(5, truncate=False)

**What you should see:** five test customers with their true `label`, a `probability` and a `prediction`. The probability is a pair: `[chance of 0, chance of 1]`, which always adds up to 1. The prediction is 1 when the chance of 1 is above 0.5. In our run the first two customers are real buyers (label 1.0) whom the model gave only about 33% and 46% and so predicted 0: mistakes. The third, with about 53%, was predicted correctly.

Want to see what the model actually received? The `features` column holds the vector of each customer:

In [ ]:
row = predictions.select("raw_features").first()["raw_features"]
print("length of the feature vector:", len(row))
print("first six values (age, income, children, children at home, house owner, cars):", row.toArray()[:6])

**What you should see:** a vector of a few dozen numbers (the six numeric features plus one position for every category of every text column), and the first six values of one test customer in their original units, such as an age around 30 to 60 and an income such as 30000.0. The model never sees the words, only these numbers.

## 6. Evaluating honestly

Three views of the same predictions:

* **accuracy**: the share of test customers predicted correctly;
* the **confusion matrix**: a 2 by 2 table of true labels against predictions, which shows *which* mistakes the model makes;
* the **AUC**: the probability that the model gives a random buyer a higher score than a random non-buyer (0.5 is guessing, 1.0 is perfect). The AUC does not depend on the 0.5 threshold, which makes it the fairest single number for comparing models.

In [ ]:
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

def report(name, predictions):
    accuracy = predictions.select(F.avg((F.col("label") == F.col("prediction")).cast("double"))).first()[0]
    print(f"{name:<24} accuracy {accuracy:.3f}   AUC {auc.evaluate(predictions):.3f}")

report("logistic regression", predictions)
(predictions.groupBy("label").pivot("prediction").count()
            .withColumnRenamed("0.0", "predicted 0").withColumnRenamed("1.0", "predicted 1")
            .orderBy("label").show())

**What you should see:** `accuracy 0.640   AUC 0.680`, and a confusion matrix with about 1,140 correct on each diagonal cell (non-buyers predicted 0, buyers predicted 1) and about 640 to 650 mistakes in each of the other two cells.

Read the matrix row by row. Of the real non-buyers (label 0.0), about 1,140 were recognised and about 650 would receive a brochure for nothing. Of the real buyers (label 1.0), about 1,140 were found and about 640 were missed.

So the model beats the baseline (64% against 50.6%), but it is far from perfect: demographics alone carry only part of the answer, which is what the weak correlations of Notebook 16 warned us about. That is a realistic result, and reporting it honestly is part of the work.

## 7. A second model, and what it relies on

A **decision tree** asks a chain of yes-or-no questions (*income above 50,000? more than one car? ...*) and can capture combinations that logistic regression cannot (for example, *young and two cars*). It also tells us which features its questions use most. Only the last stage of the pipeline changes.

In [ ]:
tree = Pipeline(stages=prepare + [DecisionTreeClassifier(maxDepth=6, seed=42)]).fit(train)
tree_predictions = tree.transform(test)
report("logistic regression", predictions)
report("decision tree (depth 6)", tree_predictions)

**What you should see:** the tree with accuracy about 0.700 and AUC about 0.634, against 0.640 and 0.680 for logistic regression.

The two measures **disagree**. The tree is right more often at the 0.5 threshold, yet it **ranks** customers worse. A tree of depth 6 has at most 64 leaves, so it gives only a few dozen distinct probabilities, and many customers tie; logistic regression gives every customer a different, smoothly varying score. If the task is *mail the 1,000 most likely buyers*, ranking is what matters and the AUC is the measure to trust. If it is *decide yes or no for each customer*, accuracy at a chosen threshold matters. Decide which question you are answering before you choose the winner (Exercise 6 tests this directly).

Now, which features does the tree use? Each feature gets an **importance**: the share of the tree's improvement that came from questions about it.

In [ ]:
attrs = tree_predictions.schema["raw_features"].metadata["ml_attr"]["attrs"]   # names set by the assembler
names = {a["idx"]: a["name"] for group in attrs.values() for a in group}
importance = tree.stages[-1].featureImportances
top = sorted(((importance[i], names[i]) for i in range(len(names)) if importance[i] > 0), reverse=True)[:8]
for value, name in top:
    print(f"{name:<40} {value:.3f}")

**What you should see:** income and the number of cars at the top (about 0.23 each), then the number of children and age (about 0.10 each), living in Australia (about 0.10), and the commute distance.

That ranking describes **this tree** and not the world: a different depth or seed changes it, and an importance is not a cause. Treat it as a reason to look further, for example by comparing the bike-buying rate across numbers of cars, not as a conclusion.

## 8. Without a label: segmenting customers

### Intuition first
Marketing also asks a question without a right answer: *what kinds of customers do we have?* **k-means** groups similar customers into *k* groups (clusters): it places *k* centres, gives each customer to the nearest centre, moves each centre to the middle of its customers, and repeats.

We describe each customer by three classic behavioural numbers, known as **RFM**: **R**ecency (days since the last order), **F**requency (number of orders) and **M**onetary value (total spend), the last one on a log scale because it is so skewed (Notebook 16). Behaviour may be used here, because we are describing customers, not predicting a future label.

We do not know the right *k*, so we try 2 to 7 and score each with the **silhouette**: how much closer each customer is to its own cluster than to the next one, from -1 to 1, higher is better.

In [ ]:
rfm = (sales.groupBy("CustomerKey")
            .agg(F.datediff(F.lit(last_day), F.max(F.to_date(F.substring("OrderDate", 1, 10)))).alias("recency"),
                 F.countDistinct("SalesOrderNumber").alias("frequency"),
                 F.log10(F.sum("SalesAmount")).alias("monetary_log")))
rfm_vec = Pipeline(stages=[VectorAssembler(inputCols=["recency", "frequency", "monetary_log"], outputCol="raw"),
                           StandardScaler(inputCol="raw", outputCol="features", withMean=True)]).fit(rfm).transform(rfm)
rfm_vec.cache()

silhouette = ClusteringEvaluator()
scores = {}
for k in range(2, 8):
    model = KMeans(k=k, seed=42).fit(rfm_vec)
    scores[k] = silhouette.evaluate(model.transform(rfm_vec))
    print(f"k = {k}: silhouette {scores[k]:.3f}")

**What you should see:** six scores between about 0.45 and 0.58, highest at **k = 4** (about 0.58).

Choose a k with a high score that the business can also use: four or five segments can be named and acted on, eleven cannot. Here the best score and a usable number agree. Let us describe the four segments.

In [ ]:
best_k = max(scores, key=scores.get)
segments = KMeans(k=best_k, seed=42).fit(rfm_vec).transform(rfm_vec)
(segments.groupBy("prediction")
         .agg(F.count("*").alias("customers"), F.round(F.avg("recency")).alias("avg_recency_days"),
              F.round(F.avg("frequency"), 2).alias("avg_orders"),
              F.round(F.avg(F.pow(10, "monetary_log"))).alias("avg_spend"))
         .orderBy(F.desc("avg_spend")).show())

**What you should see:** four segments (the numbers in `prediction` are only labels, and may come out in a different order):

* about 8,000 customers who ordered about 150 days ago, about 1.8 times, spending about 3,100 on average: **recent bike buyers**;
* about 1,600 who spent about 2,650 but last ordered well over a year ago: **lapsed bike buyers**, worth a reminder;
* 35 customers with about 20 orders each, the most recent only a few weeks ago, but only about 1,000 spent: **loyal accessory buyers**;
* about 8,900 customers with about one order and about 57 spent: **one-off small buyers**.

Each name suggests a different action. The clustering found the groups; naming them and deciding what to do is the analyst's job.

## ⚠️ Common pitfalls

* **Leakage**: a feature that is only known after the event you predict. The model looks excellent in testing and fails in use.
* **Fitting any preparation step on all the data** before splitting. The test set then leaks into training through the scaler or the indexer.
* **Trusting accuracy alone**, especially when one class is rare. Look at the confusion matrix and the AUC.
* **Reading feature importances as causes.** They describe one model, not the world.
* **Choosing k for k-means by habit.** Score several values, and ask whether the segments can be used.

## 9. Where we stand

| Step | What MLlib provides | The trap it guards against |
|---|---|---|
| Training table | DataFrame joins and aggregations | leakage: features unknown at prediction time |
| Baseline | a one-line aggregation | celebrating a model that beats nothing |
| Preparation | indexers, encoders, assembler, scaler | fitting preparation steps on the test data |
| Model | a pipeline with an estimator at the end | steps applied in different orders in training and use |
| Evaluation | evaluators, confusion matrix by `groupBy` | trusting accuracy alone |
| Segmentation | k-means with the silhouette | choosing k by habit, or clusters nobody can use |

The models were trained on one machine, but nothing in the code would change on a cluster except the `.master(...)` line. The last notebook asks the question this course has been asking all along, now about this data: did we need a cluster at all?

---
## ✏️ Exercises

## Level 1: warm-up

Small changes to the code of the notebook.

### Exercise 1 (Is the split balanced?)
Count the buyers and non-buyers in `train` and in `test` separately. Is the share of buyers about the same in both?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for name, part in [("train", train), ("test", test)]:
    counts = part.groupBy("label").count().toPandas().set_index("label")["count"]
    print(f"{name:<6} buyers {counts[1.0]:>5}   non-buyers {counts[0.0]:>5}   share of buyers {counts[1.0] / counts.sum():.1%}")
```

*7,351 buyers and 7,557 non-buyers in train (49.3% buyers), 1,781 and 1,795 in test (49.8%). A random split of a large table keeps the shares close, so the test set is a fair sample. With rare classes, check it every time.*

</details>

### Exercise 2 (The most likely buyers)
Using `predictions`, show the five test customers with the **highest** probability of buying, with their true label. (`vector_to_array("probability")[1]` reads the chance of 1.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
(predictions.withColumn("p_buy", vector_to_array("probability")[1])
            .select("CustomerKey", "label", F.round("p_buy", 3).alias("p_buy"))
            .orderBy(F.desc("p_buy"))
            .show(5))
```

*Five customers with a chance of about 0.94. Four of them really bought a bike, but the fourth (customer 11079) did not: even the most confident predictions are sometimes wrong. A probability of 0.94 means that about 6 such customers in 100 will not buy.*

</details>

### Exercise 3 (A smaller tree)
Train a decision tree with `maxDepth=3` instead of 6 and report it next to the other two models.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
small_tree = Pipeline(stages=prepare + [DecisionTreeClassifier(maxDepth=3, seed=42)]).fit(train)
report("decision tree (depth 3)", small_tree.transform(test))
report("decision tree (depth 6)", tree_predictions)
report("logistic regression", predictions)
```

*The depth-3 tree is weaker on both measures (accuracy about 0.617, AUC about 0.630). With only 8 leaves it can ask too few questions: a model that is too simple misses patterns, just as one that is too complex memorises noise.*

</details>

## Level 2: think it through

These questions are about **reasoning**. Decide first what you would need to see, then write the code that shows it.

### Exercise 4 (Better than guessing, on the test set?)
Section 3 computed the baseline on all customers. **Which number would you compare the model's 0.640 with, to be completely fair?** Compute the accuracy of *always predict the majority label of the training set* on the **test** set.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
majority = train.groupBy("label").count().orderBy(F.desc("count")).first()["label"]
baseline = test.select(F.avg((F.col("label") == majority).cast("double"))).first()[0]
print(f"majority label in train: {majority}   its accuracy on test: {baseline:.3f}")
report("logistic regression", predictions)
```

*Always predicting 0 (the more common label in train) is right on 50.2% of the test customers, so the fair comparison is 0.640 against 0.502. The baseline must be computed on the same rows as the model, with only what the training data knew.*

</details>

### Exercise 5 (Which mistake costs more?)
A brochure costs 2 euros; a missed buyer costs the shop 300 euros of profit. **Which cell of the confusion matrix matters more, and what would you change to reduce it?** Compute precision and recall for the thresholds 0.3, 0.5 and 0.7, where *precision* = the share of real buyers among those we mail, *recall* = the share of all real buyers that we mail.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
p1 = predictions.withColumn("p1", vector_to_array("probability")[1])
for t in (0.3, 0.5, 0.7):
    c = p1.select(F.sum(((F.col("p1") >= t) & (F.col("label") == 1)).cast("int")).alias("tp"),
                  F.sum(((F.col("p1") >= t) & (F.col("label") == 0)).cast("int")).alias("fp"),
                  F.sum(((F.col("p1") < t) & (F.col("label") == 1)).cast("int")).alias("fn")).first()
    print(f"threshold {t}: precision {c.tp / (c.tp + c.fp):.3f}   recall {c.tp / (c.tp + c.fn):.3f}   mailed {c.tp + c.fp}")
```

*A missed buyer costs 150 times more than a wasted brochure, so recall matters most here. Lowering the threshold to 0.3 mails 3,038 customers and finds about 91% of the real buyers, at a precision of only 0.53; the threshold 0.7 mails 426 people, 74% of them buyers, but misses more than four buyers in five. The right threshold is a business decision about the cost of each kind of mistake, not a technical default.*

</details>

### Exercise 6 (Accuracy or ranking?)
The tree has the better accuracy, logistic regression the better AUC. Marketing can mail exactly **1,000** test customers. **Which model would you choose, and what single number would prove it?** For each model, take the 1,000 customers with the highest probability of buying and compute the share of real buyers among them.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def buyers_in_top(preds, n=1000):
    top = (preds.withColumn("p1", vector_to_array("probability")[1])
                .orderBy(F.desc("p1"), "CustomerKey").limit(n))
    return top.select(F.avg("label")).first()[0]

print(f"logistic regression: {buyers_in_top(predictions):.1%} of the top 1,000 are real buyers")
print(f"decision tree:       {buyers_in_top(tree_predictions):.1%} of the top 1,000 are real buyers")
```

*Almost a tie: 70.7% real buyers in the top 1,000 for logistic regression, 69.7% for the tree. The AUC gap is about the ranking of the whole list; at the very top both models are equally good. So for this mailing neither model clearly wins, and a single number taken from the real decision settled what the two summary measures left open. Measure what the decision needs.*

</details>

## Level 3: technical

Longer tasks: more models, tuning, and a leakage experiment.

### Exercise 7 (A random forest)
Replace the last stage with a `RandomForestClassifier(numTrees=50, maxDepth=8, seed=42)` and compare its accuracy and AUC with the two models above.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from pyspark.ml.classification import RandomForestClassifier
forest = Pipeline(stages=prepare + [RandomForestClassifier(numTrees=50, maxDepth=8, seed=42)]).fit(train)
report("random forest", forest.transform(test))
report("decision tree (depth 6)", tree_predictions)
report("logistic regression", predictions)
```

*The forest wins on both measures (about 0.728 accuracy and 0.799 AUC). Averaging many trees grown on different samples smooths the scores, so the forest ranks well too. Whether the gain is worth fifty times the training work is a judgement to make, and to report.*

</details>

### Exercise 8 (Is it leakage?)
Add the customer's **number of orders** to the features (join it from `sales`), retrain the logistic regression and report the AUC. Then explain in two sentences why the improvement must not be reported as a better model for finding new bike buyers.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
orders = sales.groupBy("CustomerKey").agg(F.countDistinct("SalesOrderNumber").cast("double").alias("orders"))
train_l, test_l = [d.join(orders, "CustomerKey") for d in (train, test)]
leaky_assembler = VectorAssembler(inputCols=numeric + ["orders"] + [c + "_vec" for c in categorical],
                                  outputCol="raw_features")
leaky = Pipeline(stages=indexers + [encoder, leaky_assembler, scaler, LogisticRegression(maxIter=50)]).fit(train_l)
report("with number of orders", leaky.transform(test_l))
report("demographics only", predictions)
```

*The AUC rises from about 0.680 to about 0.723. But the number of orders is known only after the customer has bought, and buying a bicycle is part of it, so the feature carries the answer into the question. For a new prospect it is zero or unknown, and the better score would vanish the day the model was used.*

</details>

### Exercise 9 (Tuning with cross-validation, a little harder)
Use `CrossValidator` with 3 folds and a `ParamGridBuilder` over `regParam` in `[0.0, 0.01, 0.1]` for the logistic regression, choosing by AUC. Report the best `regParam` and the test AUC of the best model. (This takes a minute or two.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
lr = LogisticRegression(maxIter=50)
grid = ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.01, 0.1]).build()
cv = CrossValidator(estimator=Pipeline(stages=prepare + [lr]), estimatorParamMaps=grid,
                    evaluator=auc, numFolds=3, seed=42).fit(train)
print("mean AUC per setting:", [round(m, 4) for m in cv.avgMetrics])
print("best regParam:", cv.bestModel.stages[-1].getRegParam())
report("tuned logistic", cv.bestModel.transform(test))
```

*The three settings score almost the same (about 0.70), the best is regParam 0.0, and the tuned model is the model we already had. Tuning cannot add information the features do not contain. Cross-validation uses only the training data, and the whole pipeline is cross-validated, preparation included: that is what keeps each fold honest.*

</details>

## 🔑 Key takeaways

- A classifier learns from **features** and a **label**; it is graded on a **test set** it never saw.
- Build the **training table** at the unit you predict about, and exclude anything not known at prediction time: that is **leakage**.
- Always compare with a **baseline**; with rare classes, prefer precision, recall and the **AUC** to accuracy.
- An MLlib **pipeline** fits every preparation step on the training data only and applies it identically everywhere.
- Accuracy and AUC answer different questions; choose the measure that matches the decision.
- Feature importances describe a model, not the world; they are leads, not causes.
- **k-means** on RFM features gives segments; choose k with the **silhouette** and with what the business can use.

**Next:** *Notebook 18: Measuring Scale*, the last notebook, where pandas, SQLite and Spark answer the same question on growing copies of this data, and we find where each one stops being the right tool.

In [ ]:
spark.stop()

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*